In [0]:
# =============================================================================
# F1-Pulse | Bronze Layer — Safe Ingestion
# Notebook: 01_Ingest_F1_Data
# Author:   Jafar891
# Updated:  2026
#
# Ingests raw F1 data from the OpenF1 REST API into Delta Lake Bronze tables.
# Designed for Databricks (Serverless). Idempotent — safe to re-run.
# =============================================================================

import sys, importlib
import logging

sys.path.append("/Workspace/Users/jafar.gahramanov@gmail.com/F1-Pulse")

# Reload modules so Databricks picks up edits without a cluster restart
import modules.api_client    as api_client
import modules.f1_helpers    as f1_helpers
import modules.bronze_helpers as bronze_helpers
importlib.reload(api_client)
importlib.reload(f1_helpers)
importlib.reload(bronze_helpers)

from modules.api_client     import fetch_with_retry
from modules.f1_helpers     import get_latest_race_session, pdf_to_spark
from modules.bronze_helpers import write_bronze

from config.config import (
    YEAR, CATALOG, BRONZE, BASE_URL,
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("f1_pulse.bronze")


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------

def ingest_bronze() -> None:

    log.info("=" * 60)
    log.info(f"F1-Pulse Bronze Ingestion — {YEAR}")
    log.info("=" * 60)

    # ------------------------------------------------------------------
    # 1. Sessions
    # ------------------------------------------------------------------
    sessions_url = f"{BASE_URL}/sessions?year={YEAR}&session_type=Race"
    log.info(f"\n[1/5] Ingesting sessions ({YEAR}) …")

    session_data = fetch_with_retry(sessions_url)

    if not session_data:
        raise RuntimeError("Cannot proceed — sessions endpoint failed.")

    df_sessions = pdf_to_spark(spark, session_data, sessions_url)

    if df_sessions:
        write_bronze(df_sessions, CATALOG, BRONZE, f"raw_sessions_{YEAR}")

    # ------------------------------------------------------------------
    # 2. Resolve latest race session
    # ------------------------------------------------------------------
    latest_session = get_latest_race_session(session_data)

    if not latest_session:
        raise RuntimeError("Could not determine latest race session.")

    session_key  = latest_session["session_key"]
    session_name = latest_session.get("session_name", "Unknown")

    log.info(f"\n  Latest race session → key={session_key}  name={session_name}")

    # ------------------------------------------------------------------
    # 3. Race endpoints
    # ------------------------------------------------------------------
    race_endpoints = {
        f"raw_drivers_{YEAR}":
            f"{BASE_URL}/drivers?session_key={session_key}",

        f"raw_laps_{YEAR}":
            f"{BASE_URL}/laps?session_key={session_key}",

        f"raw_telemetry_{YEAR}":
            f"{BASE_URL}/car_data?session_key={session_key}&speed>250",
    }

    results = {"success": [], "failed": []}

    # ------------------------------------------------------------------
    # 4. Ingest race datasets
    # ------------------------------------------------------------------
    for i, (table_name, url) in enumerate(race_endpoints.items(), start=2):

        log.info(f"\n[{i}/5] Ingesting {table_name} …")

        data = fetch_with_retry(url)
        if data is None:
            results["failed"].append(table_name)
            continue

        df = pdf_to_spark(spark, data, url)
        if df is None:
            results["failed"].append(table_name)
            continue

        try:
            write_bronze(df, CATALOG, BRONZE, table_name)
            results["success"].append(table_name)
        except Exception:
            results["failed"].append(table_name)

    # ------------------------------------------------------------------
    # 5. Summary
    # ------------------------------------------------------------------
    log.info("\n" + "=" * 60)
    log.info("INGESTION SUMMARY")
    log.info("=" * 60)
    log.info(f"  Session key : {session_key} ({session_name})")
    log.info(f"  ✅ Success  : {len(results['success'])} tables")
    for t in results["success"]:
        log.info(f"      → {CATALOG}.{BRONZE}.{t}")

    if results["failed"]:
        log.warning(f"  ❌ Failed   : {len(results['failed'])} tables")
        for t in results["failed"]:
            log.warning(f"      → {t}")

    log.info("=" * 60)

    if results["failed"]:
        raise RuntimeError(
            f"Ingestion completed with errors: {results['failed']}"
        )


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

ingest_bronze()